## HYBRIED-RETRIEVER (combining Dense & Sparse)

In [4]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document

In [5]:
from langchain_core.documents import Document

sample_docs = [
    Document(
        page_content="""
        FastAPI is a modern Python web framework used for building APIs.
        It provides automatic API documentation using Swagger UI and ReDoc.
        FastAPI supports asynchronous programming and uses Pydantic for
        request validation and data serialization.
        """,
        metadata={"source": "fastapi.txt", "topic": "FastAPI"}
    ),

    Document(
        page_content="""
        JWT authentication is commonly used to secure web applications.
        After successful login, the server generates a JSON Web Token.
        The client sends this token with future requests to access protected
        resources.
        """,
        metadata={"source": "jwt.txt", "topic": "Authentication"}
    ),

    Document(
        page_content="""
        Machine Learning allows computers to learn patterns from data.
        Supervised learning uses labeled datasets for classification and
        regression, while unsupervised learning discovers patterns in
        unlabeled data.
        """,
        metadata={"source": "ml.txt", "topic": "Machine Learning"}
    ),

    Document(
        page_content="""
        Retrieval Augmented Generation, or RAG, combines document retrieval
        with large language models. Documents are converted into embeddings
        and stored in a vector database. Relevant documents are retrieved
        before the language model generates an answer.
        """,
        metadata={"source": "rag.txt", "topic": "RAG"}
    ),

    Document(
        page_content="""
        Vector databases store numerical representations called embeddings.
        They are designed to perform similarity searches and retrieve
        documents that are semantically related to a query.
        """,
        metadata={"source": "vector-db.txt", "topic": "Vector Database"}
    ),

    Document(
        page_content="""
        Python is a popular programming language used in web development,
        data science, machine learning, automation, and artificial
        intelligence. Its simple syntax makes it easy to learn.
        """,
        metadata={"source": "python.txt", "topic": "Python"}
    )
]

embedding = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
dense_vectorestore = FAISS.from_documents(sample_docs,embedding)
dense_retriever = dense_vectorestore.as_retriever()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
sparse_retriever = BM25Retriever.from_documents(sample_docs)
sparse_retriever.k = 3

hybried_retriever = EnsembleRetriever(
    retrievers=[dense_retriever,sparse_retriever],
    weights=[0.7,0.3]
)

hybried_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001EFBB3697F0>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001EFBB283ED0>, k=3)], weights=[0.7, 0.3])

In [21]:
query = "How are numerical representations used to find similar documents?"
results = hybried_retriever.invoke(query)


In [22]:
for i, doc in enumerate(results):
    print(f"\nDocument {i+1}:\n{doc.page_content}")


Document 1:

        Vector databases store numerical representations called embeddings.
        They are designed to perform similarity searches and retrieve
        documents that are semantically related to a query.
        

Document 2:

        Retrieval Augmented Generation, or RAG, combines document retrieval
        with large language models. Documents are converted into embeddings
        and stored in a vector database. Relevant documents are retrieved
        before the language model generates an answer.
        

Document 3:

        Machine Learning allows computers to learn patterns from data.
        Supervised learning uses labeled datasets for classification and
        regression, while unsupervised learning discovers patterns in
        unlabeled data.
        

Document 4:

        FastAPI is a modern Python web framework used for building APIs.
        It provides automatic API documentation using Swagger UI and ReDoc.
        FastAPI supports asynchronous progr

###  RAG PIPELINE WITH hyBRIED retriever

In [25]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

In [27]:
prompt = PromptTemplate.from_template(
    """Answer the question based on the context below.
    Context:
    {context} 
    
    Question:
    {input}   
    """    
)

llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    temperature=0
)
llm

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14', 'langchain-google-genai': '4.3.2'}}, output_version=None, profile={'name': 'Gemini Flash Latest', 'release_date': '2026-05-19', 'last_updated': '2026-05-19', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-flash-latest', temperature=0.0, client=<google.genai.client.Client object at 0x000001F0327B3610>, default_metadata=(), model_kwargs={})

In [28]:
document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt
)

rag_chain = create_retrieval_chain(
    retriever=hybried_retriever,
    combine_docs_chain=document_chain
)

In [32]:
query = {"input": "how can i build an app using LLMs"}

response = rag_chain.invoke(query)

print("\n" + "=" * 60)
print("QUESTION")
print("=" * 60)
print(query["input"])

print("\n" + "=" * 60)
print("ANSWER")
print("=" * 60)
print(response["answer"])

print("\n" + "=" * 60)
print("RETRIEVED DOCUMENTS")
print("=" * 60)

for i, doc in enumerate(response["context"], start=1):
    print(f"\n--- Document {i} ---")
    print("Source :", doc.metadata.get("source"))
    print("Topic  :", doc.metadata.get("topic"))
    print("Content:")
    print(doc.page_content)


QUESTION
how can i build an app using LLMs

ANSWER
Based on the provided context, you can build an application using Large Language Models (LLMs) by implementing **Retrieval Augmented Generation (RAG)**:

1. **Convert Documents to Embeddings:** Convert your documents into numerical representations called embeddings.
2. **Store in a Vector Database:** Store these embeddings in a vector database, which is designed to perform similarity searches for semantically related content.
3. **Retrieve Relevant Documents:** When a query is made, retrieve the most relevant documents from the vector database.
4. **Generate an Answer:** Provide the retrieved documents to the large language model before it generates an answer.

*(Note: To turn this into a web application/API, the context also mentions using **FastAPI** to build the API and **JWT authentication** to secure protected resources.)*

RETRIEVED DOCUMENTS

--- Document 1 ---
Source : fastapi.txt
Topic  : FastAPI
Content:

        FastAPI is 